### Modell anzeigen

In [38]:
from core.preprocess import LLM_MODEL, OLLAMA_URL
print("LLM_MODEL =", LLM_MODEL)
print("OLLAMA_URL =", OLLAMA_URL)

LLM_MODEL = gpt-oss:120b-cloud
OLLAMA_URL = http://localhost:11434


### 1) URL-Erweiterung automatisch erzeugen

In [66]:
import re

# base_url = "https://datenbank2.deutscher-nachhaltigkeitskodex.de/Profile/MainMenuHandler/1_1?company=18552&year=2024&lang=de&culture=de"
base_url = "data/url_sources/dnk_2024_individually.txt"


# Definition aller gewünschten MainMenuHandler IDs
ranges = {
    1: range(1, 2),     # 1_1
    2: range(1, 5),     # 2_1 ... 2_4
    3: range(1, 11),    # 3_1 ... 3_10
    4: range(1, 6),     # 4_1 ... 4_5
    5: range(1, 13)     # 5_1 ... 5_12
}

# Basis-URL ohne MainMenuHandler extrahieren
prefix = re.sub(r"MainMenuHandler/\d+_\d+", "MainMenuHandler/{}", base_url)

# Liste aller erzeugten URLs
urls = []

for chapter, numbers in ranges.items():
    for number in numbers:
        urls.append(prefix.format(f"{chapter}_{number}"))

# Ausgabe
for u in urls:
    print(u)


https://datenbank2.deutscher-nachhaltigkeitskodex.de/Profile/MainMenuHandler/1_1?company=18552&year=2024&lang=de&culture=de
https://datenbank2.deutscher-nachhaltigkeitskodex.de/Profile/MainMenuHandler/2_1?company=18552&year=2024&lang=de&culture=de
https://datenbank2.deutscher-nachhaltigkeitskodex.de/Profile/MainMenuHandler/2_2?company=18552&year=2024&lang=de&culture=de
https://datenbank2.deutscher-nachhaltigkeitskodex.de/Profile/MainMenuHandler/2_3?company=18552&year=2024&lang=de&culture=de
https://datenbank2.deutscher-nachhaltigkeitskodex.de/Profile/MainMenuHandler/2_4?company=18552&year=2024&lang=de&culture=de
https://datenbank2.deutscher-nachhaltigkeitskodex.de/Profile/MainMenuHandler/3_1?company=18552&year=2024&lang=de&culture=de
https://datenbank2.deutscher-nachhaltigkeitskodex.de/Profile/MainMenuHandler/3_2?company=18552&year=2024&lang=de&culture=de
https://datenbank2.deutscher-nachhaltigkeitskodex.de/Profile/MainMenuHandler/3_3?company=18552&year=2024&lang=de&culture=de
https://

### 2) Inhalt der Erweiterungen als MD-Format speichern

In [59]:
import requests
from pathlib import Path
from bs4 import BeautifulSoup
from markdownify import markdownify as mdify
from utils.logger import logger

URL_FILE = Path("data/url_sources/trash.txt")
OUTPUT_DIR = Path("docs_tests/raw/markdown/dnk_md_2024")
OUTPUT_DIR.mkdir(exist_ok=True)

# URLs aus Datei-(URL_FILE) lesen
urls = [u.strip() for u in URL_FILE.read_text(encoding="utf-8").splitlines() if u.strip()]

logger.info("HTML laden und in Markdown konvertieren …")

# jede 32 URLs → 1 Datei
GROUP_SIZE = 32
groups = [urls[i:i + GROUP_SIZE] for i in range(0, len(urls), GROUP_SIZE)]

for file_index, group in enumerate(groups, start=1):

    output_file = OUTPUT_DIR / f"dnk_datei_2024_{file_index}.md"
    output_file.write_text("", encoding="utf-8")

    for url in group:

        try:
            # HTML abrufen
            r = requests.get(url, headers={"User-Agent": "MD-Scraper"})
            if "html" not in r.headers.get("Content-Type", "").lower():
                logger.warning(f"⚠️ Übersprungen (kein HTML): {url}")
                continue

            # HTML parsen
            soup = BeautifulSoup(r.text, "html.parser")

            # HTML bereinigen
            for tag in soup.find_all("a"): tag.unwrap()      # Links entfernen: <a> bleibt Text
            for tag in soup.find_all("img"): tag.decompose() # Bilder entfernen: komplette <img>-Tags löschen

            # Bereinigtes HTML → Markdown, dann speichern
            markdown = mdify(str(soup), heading_style="ATX")

            output_file.write_text(
                output_file.read_text(encoding="utf-8") +
                f"\n\n" + markdown,encoding="utf-8"
            )
            logger.debug(f"    - Gespeichert: {url}")

        except Exception:
            logger.exception("⚠️ Fehler beim Verarbeiten der URL")

logger.info("HTML laden und in Markdown konvertierung ist abgeschlossen.")

### Test Chunking

In [ ]:
# utils/chunking.py
from langchain_text_splitters import MarkdownHeaderTextSplitter
from utils.logger import logger

def chunk_documents(docs):
    logger.info("Chunking beginnt ...")

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("#", "H1"), ("##", "H2"), ("###", "H3")],
        strip_headers=False
    )

    chunks = [] # An empty list is created in which all generated chunks are collected.
    for doc in docs:
        for chunk in splitter.split_text(doc.page_content):
            chunk.metadata["source"] = doc.metadata.get("source") # Add  metadata `source` for example: `dokument_1.md`
            chunks.append(chunk) # Save the new chunk

    logger.info("Chunking abgeschlossen.")
    return chunks


In [104]:
markdown_document = """
    # Das ist Header 1 üüü
    text_Header1

    ## Das ist Header 2
    text_Header2

    ### Das ist Header 3
    text_Header3
"""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)
md_header_splits = markdown_splitter.split_text(markdown_document)
md_header_splits

[Document(metadata={'Header 1': 'Das ist Header 1 üüü'}, page_content='# Das ist Header 1 üüü\ntext_Header1'),
 Document(metadata={'Header 1': 'Das ist Header 1 üüü', 'Header 2': 'Das ist Header 2'}, page_content='## Das ist Header 2\ntext_Header2'),
 Document(metadata={'Header 1': 'Das ist Header 1 üüü', 'Header 2': 'Das ist Header 2', 'Header 3': 'Das ist Header 3'}, page_content='### Das ist Header 3\ntext_Header3')]